In [13]:
from preprocessing_functions import *
from processing import *

data_P1_pre_training = scipy.io.loadmat(r"C:\Users\rodri\OneDrive\Documentos\g.tec hackaton\Hackaton_Rehab_Data_Analysis\stroke-rehab\P1_pre_training.mat")
data_P1_pre_test = scipy.io.loadmat(r"C:\Users\rodri\OneDrive\Documentos\g.tec hackaton\Hackaton_Rehab_Data_Analysis\stroke-rehab\P1_pre_test.mat")
data_P1_post_training = scipy.io.loadmat(r"C:\Users\rodri\OneDrive\Documentos\g.tec hackaton\Hackaton_Rehab_Data_Analysis\stroke-rehab\P1_post_training.mat")
data_P1_post_test = scipy.io.loadmat(r"C:\Users\rodri\OneDrive\Documentos\g.tec hackaton\Hackaton_Rehab_Data_Analysis\stroke-rehab\P1_post_test.mat")

In [14]:
fs = data_P1_pre_training["fs"][0][0] 

trig_pre_training = data_P1_pre_training['trig']
y_pre_training = data_P1_pre_training['y']

trig_pre_training = trig_pre_training.T
print(trig_pre_training.shape)

trig_pre_test = data_P1_pre_test['trig']
y_pre_test = data_P1_pre_test['y']

print(trig_pre_test.shape)

trig_post_training = data_P1_post_training['trig']
y_post_training = data_P1_post_training['y']    

print(trig_post_training.shape)

trig_post_test = data_P1_post_test['trig']
y_post_test = data_P1_post_test['y']    

print(trig_post_test.shape)

(271816, 1)
(204560, 1)
(197343, 1)
(194088, 1)


In [15]:
trigger_events_time_pre_training = get_trigger_onsets(trig_pre_training, fs)
trigger_events_time_pre_test = get_trigger_onsets(trig_pre_test, fs)
trigger_events_time_post_training = get_trigger_onsets(trig_post_training, fs)
trigger_events_time_post_test = get_trigger_onsets(trig_post_test, fs)

In [16]:
y_pre_training_trimmed = trim_data(y_pre_training, fs, start_time = trigger_events_time_pre_training[0] - 1)
y_pre_test_trimmed = trim_data(y_pre_test, fs, start_time = trigger_events_time_pre_test[0] - 1)
y_post_training_trimmed = trim_data(y_post_training, fs, start_time = trigger_events_time_post_training[0] - 1)
y_post_test_trimmed = trim_data(y_post_test, fs, start_time = trigger_events_time_post_test[0] - 1)


trigger_events_trimmed_pre_training = trim_data(trig_pre_training, fs, start_time = trigger_events_time_pre_training[0] - 1)
trigger_events_trimmed_pre_test = trim_data(trig_pre_test, fs, start_time = trigger_events_time_pre_test[0] - 1)
trigger_events_trimmed_post_training = trim_data(trig_post_training, fs, start_time = trigger_events_time_post_training[0] - 1)
trigger_events_trimmed_post_test = trim_data(trig_post_test, fs, start_time = trigger_events_time_post_test[0] - 1)

t_trimmed = np.arange(y_pre_training.shape[0]) / fs

In [17]:
y_pre_training_trimmed = notch_filter(y_pre_training_trimmed, fs, notch_freq = 50)
y_pre_training_trimmed = notch_filter(y_pre_training_trimmed, fs, notch_freq = 60)
y_pre_training_trimmed = bandpass_filter(y_pre_training_trimmed, fs, lowcut=1, highcut=40, order=4)

print(y_pre_training_trimmed.shape)

y_pre_test_trimmed = notch_filter(y_pre_test_trimmed, fs, notch_freq = 50)
y_pre_test_trimmed = notch_filter(y_pre_test_trimmed, fs, notch_freq = 60)
y_pre_test_trimmed = bandpass_filter(y_pre_test_trimmed, fs, lowcut=1, highcut=40, order=4)

y_post_training_trimmed = notch_filter(y_post_training_trimmed, fs, notch_freq = 50)
y_post_training_trimmed = notch_filter(y_post_training_trimmed, fs, notch_freq = 60)
y_post_training_trimmed = bandpass_filter(y_post_training_trimmed, fs, lowcut=1, highcut=40, order=4)

y_post_test_trimmed = notch_filter(y_post_test_trimmed, fs, notch_freq = 50)
y_post_test_trimmed = notch_filter(y_post_test_trimmed, fs, notch_freq = 60)
y_post_test_trimmed = bandpass_filter(y_post_test_trimmed, fs, lowcut=1, highcut=40, order=4)    

(206234, 16)


In [18]:
print(y_pre_training.shape)

right_hand_epochs_p1_pre_training, left_hand_epochs_p1_pre_training = epoch_data(y_pre_training_trimmed, trigger_events_trimmed_pre_training, fs, time_after=2.0, period_of_interest=6.0)
right_hand_epochs_p1_pre_test, left_hand_epochs_p1_pre_test = epoch_data(y_pre_test_trimmed, trigger_events_trimmed_pre_test, fs, time_after=2.0, period_of_interest=6.0)
right_hand_epochs_p1_post_training, left_hand_epochs_p1_post_training = epoch_data(y_post_training_trimmed, trigger_events_trimmed_post_training, fs, time_after=2.0, period_of_interest=6.0)
right_hand_epochs_p1_post_test, left_hand_epochs_p1_post_test = epoch_data(y_post_test_trimmed, trigger_events_trimmed_post_test, fs, time_after=2.0, period_of_interest=6.0)

print(right_hand_epochs_p1_pre_training.shape)

(271816, 16)
(40, 1536, 16)


In [19]:
CHANNEL_NAMES = ['FC3','FCz','FC4','C5','C3','C1','Cz','C2','C4','C6','CP3','CP1','CPz','CP2','CP4','Pz']

info = mne.create_info(ch_names=CHANNEL_NAMES, sfreq=fs, ch_types='eeg')

montage = mne.channels.make_standard_montage('standard_1020')
info.set_montage(montage)

<Info | 8 non-empty values
 bads: []
 ch_names: FC3, FCz, FC4, C5, C3, C1, Cz, C2, C4, C6, CP3, CP1, CPz, CP2, ...
 chs: 16 EEG
 custom_ref_applied: False
 dig: 19 items (3 Cardinal, 16 EEG)
 highpass: 0.0 Hz
 lowpass: 128.0 Hz
 meas_date: unspecified
 nchan: 16
 projs: []
 sfreq: 256.0 Hz
>

In [ ]:
right_hand_epochs_p1_pre_training = autoreject_epochs(right_hand_epochs_p1_pre_training, info=info)
left_hand_epochs_p1_pre_training = autoreject_epochs(left_hand_epochs_p1_pre_training, info=info)
right_hand_epochs_p1_pre_test = autoreject_epochs(right_hand_epochs_p1_pre_test, info=info)
left_hand_epochs_p1_pre_test = autoreject_epochs(left_hand_epochs_p1_pre_test, info=info)
right_hand_epochs_p1_post_training = autoreject_epochs(right_hand_epochs_p1_post_training, info=info)
left_hand_epochs_p1_post_training = autoreject_epochs(left_hand_epochs_p1_post_training, info=info)
right_hand_epochs_p1_post_test = autoreject_epochs(right_hand_epochs_p1_post_test, info=info)
left_hand_epochs_p1_post_test = autoreject_epochs(left_hand_epochs_p1_post_test, info=info)

Not setting metadata
40 matching events found
No baseline correction applied
0 projection items activated
Running autoreject on ch_type=eeg


100%|██████████| Creating augmented epochs : 16/16 [00:00<00:00,  100.41it/s]
100%|██████████| Computing thresholds ... : 16/16 [00:10<00:00,    1.52it/s]











100%|██████████| Repairing epochs : 40/40 [00:00<00:00,  206.88it/s]












100%|██████████| Repairing epochs : 40/40 [00:00<00:00,  178.76it/s]






















100%|██████████| Fold : 10/10 [00:01<00:00,    5.87it/s]
















100%|██████████| Repairing epochs : 40/40 [00:00<00:00,  139.54it/s]






















100%|██████████| Fold : 10/10 [00:01<00:00,    8.11it/s]














100%|██████████| Repairing epochs : 40/40 [00:00<00:00,  157.53it/s]


















100%|██████████| Fold : 10/10 [00:00<00:00,   61.51it/s]
100%|██████████| n_interp : 3/3 [00:04<00:00,    1.34s/it]






Estimated consensus=0.90 and n_interpolate=1
















100%|██████████| Repairing epochs : 40/40 [00:00<00:00,  163.96it/s]

No bad epochs were found for your data. Returning a copy of the data you wanted to clean. Interpolation may have been done.
AutoReject: 40 → 40 epochs
Not setting metadata


...

100%|██████████| Creating augmented epochs : 16/16 [00:00<00:00,   86.91it/s]
100%|██████████| Computing thresholds ... : 16/16 [00:09<00:00,    1.75it/s]












100%|██████████| Repairing epochs : 40/40 [00:00<00:00,  188.62it/s]














100%|██████████| Repairing epochs : 40/40 [00:00<00:00,  162.21it/s]






















100%|██████████| Fold : 10/10 [00:01<00:00,    7.97it/s]












100%|██████████| Repairing epochs : 40/40 [00:00<00:00,  182.11it/s]






















100%|██████████| Fold : 10/10 [00:01<00:00,    9.78it/s]

c:\Users\rodri\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\rodri\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)











100%|██████████| Repairing epochs : 40/4





Estimated consensus=0.50 and n_interpolate=1















100%|██████████| Repairing epochs : 40/40 [00:00<00:00,  159.79it/s]

Dropped 4 epochs: 3, 8, 23, 38


AutoReject: 40 → 36 epochs


In [ ]:
import json
import numpy as np

data_to_save = {
    "right_hand_epochs": right_hand_epochs_p1_pre_training.tolist(),
    "left_hand_epochs": left_hand_epochs_p1_pre_training.tolist(),
    "labels": {
        "right": -1,
        "left": 1
    },
    "shape_info": {
        "format": "(epochs, samples, channels)",
        "right_shape": right_hand_epochs_p1_pre_training.shape,
        "left_shape": left_hand_epochs_p1_pre_training.shape
    }
}

with open("P1_pre_training_epochs_clean.json", "w") as f:
    json.dump(data_to_save, f)

print("Saved successfully!")


data_to_save = {
    "right_hand_epochs": right_hand_epochs_p1_pre_test.tolist(),
    "left_hand_epochs": left_hand_epochs_p1_pre_test.tolist(),
    "labels": {
        "right": -1,
        "left": 1
    },
    "shape_info": {
        "format": "(epochs, samples, channels)",
        "right_shape": right_hand_epochs_p1_pre_test.shape,
        "left_shape": left_hand_epochs_p1_pre_test.shape
    }
}

with open("P1_pre_test_epochs_clean.json", "w") as f:
    json.dump(data_to_save, f)

print("Saved successfully!")


data_to_save = {
    "right_hand_epochs": right_hand_epochs_p1_post_training.tolist(),
    "left_hand_epochs": left_hand_epochs_p1_post_training.tolist(),
    "labels": {
        "right": -1,
        "left": 1
    },
    "shape_info": {
        "format": "(epochs, samples, channels)",
        "right_shape": right_hand_epochs_p1_post_training.shape,
        "left_shape": left_hand_epochs_p1_post_training.shape
    }
}

with open("P1_post_training_epochs_clean.json", "w") as f:
    json.dump(data_to_save, f)

print("Saved successfully!")


data_to_save = {
    "right_hand_epochs": right_hand_epochs_p1_post_test.tolist(),
    "left_hand_epochs": left_hand_epochs_p1_post_test.tolist(),
    "labels": {
        "right": -1,
        "left": 1
    },
    "shape_info": {
        "format": "(epochs, samples, channels)",
        "right_shape": right_hand_epochs_p1_post_test.shape,
        "left_shape": left_hand_epochs_p1_post_test.shape
    }
}

with open("P1_post_test_epochs_clean.json", "w") as f:
    json.dump(data_to_save, f)

print("Saved successfully!")

Saved successfully!
Saved successfully!
Saved successfully!
Saved successfully!


In [ ]:
mu_left_pre_training = get_band_power(left_hand_epochs_p1_pre_training, fs=256, low=8, high=12)
mu_right_pre_training = get_band_power(right_hand_epochs_p1_pre_training, fs=256, low=8, high=12)

beta_left_pre_training = get_band_power(left_hand_epochs_p1_pre_training, fs=256, low=13, high=30)
beta_right_pre_training = get_band_power(right_hand_epochs_p1_pre_training, fs=256, low=13, high=30)



mu_left_pre_test = get_band_power(left_hand_epochs_p1_pre_test, fs=256, low=8, high=12)
mu_right_pre_test = get_band_power(right_hand_epochs_p1_pre_test, fs=256, low=8, high=12)

beta_left_pre_test = get_band_power(left_hand_epochs_p1_pre_test, fs=256, low=13, high=30)
beta_right_pre_test = get_band_power(right_hand_epochs_p1_pre_test, fs=256, low=13, high=30)



mu_left_post_training = get_band_power(left_hand_epochs_p1_post_training, fs=256, low=8, high=12)
mu_right_post_training = get_band_power(right_hand_epochs_p1_post_training, fs=256, low=8, high=12)

beta_left_post_training = get_band_power(left_hand_epochs_p1_post_training, fs=256, low=13, high=30)
beta_right_post_training = get_band_power(right_hand_epochs_p1_post_training, fs=256, low=13, high=30)



mu_left_post_test = get_band_power(left_hand_epochs_p1_post_test, fs=256, low=8, high=12)
mu_right_post_test = get_band_power(right_hand_epochs_p1_post_test, fs=256, low=8, high=12)

beta_left_post_test = get_band_power(left_hand_epochs_p1_post_test, fs=256, low=13, high=30)
beta_right_post_test = get_band_power(right_hand_epochs_p1_post_test, fs=256, low=13, high=30)

In [ ]:
ref_right_pre_training, ref_left_pre_training = epoch_data(
    y_pre_training_trimmed,
    trigger_events_trimmed_pre_training,
    fs,
    time_after=-1.0,
    period_of_interest=1.0
)

mu_erd_left_pre_training = calculate_erd_ers(left_hand_epochs_p1_pre_training, ref_left_pre_training, fs, 8, 12)
mu_erd_right_pre_training = calculate_erd_ers(right_hand_epochs_p1_pre_training, ref_right_pre_training, fs, 8, 12)

beta_erd_left_pre_training = calculate_erd_ers(left_hand_epochs_p1_pre_training, ref_left_pre_training, fs, 13, 30)
beta_erd_right_pre_training = calculate_erd_ers(right_hand_epochs_p1_pre_training, ref_right_pre_training, fs, 13, 30)



ref_right_pre_test, ref_left_pre_test = epoch_data(
    y_pre_test_trimmed,
    trigger_events_trimmed_pre_test,
    fs,
    time_after=-1.0,
    period_of_interest=1.0
)

mu_erd_left_pre_test = calculate_erd_ers(left_hand_epochs_p1_pre_test, ref_left_pre_test, fs, 8, 12)
mu_erd_right_pre_test = calculate_erd_ers(right_hand_epochs_p1_pre_test, ref_right_pre_test, fs, 8, 12)

beta_erd_left_pre_test = calculate_erd_ers(left_hand_epochs_p1_pre_test, ref_left_pre_test, fs, 13, 30)
beta_erd_right_pre_test = calculate_erd_ers(right_hand_epochs_p1_pre_test, ref_right_pre_test, fs, 13, 30)



ref_right_post_training, ref_left_post_training = epoch_data(
    y_post_training_trimmed,
    trigger_events_trimmed_post_training,
    fs,
    time_after=-1.0,
    period_of_interest=1.0
)

mu_erd_left_post_training = calculate_erd_ers(left_hand_epochs_p1_post_training, ref_left_post_training, fs, 8, 12)
mu_erd_right_post_training = calculate_erd_ers(right_hand_epochs_p1_post_training, ref_right_post_training, fs, 8, 12)

beta_erd_left_post_training = calculate_erd_ers(left_hand_epochs_p1_post_training, ref_left_post_training, fs, 13, 30)
beta_erd_right_post_training = calculate_erd_ers(right_hand_epochs_p1_post_training, ref_right_post_training, fs, 13, 30)



ref_right_post_test, ref_left_post_test = epoch_data(
    y_post_test_trimmed,
    trigger_events_trimmed_post_test,
    fs,
    time_after=-1.0,
    period_of_interest=1.0
)

mu_erd_left_post_test = calculate_erd_ers(left_hand_epochs_p1_post_test, ref_left_post_test, fs, 8, 12)
mu_erd_right_post_test = calculate_erd_ers(right_hand_epochs_p1_post_test, ref_right_post_test, fs, 8, 12)

beta_erd_left_post_test = calculate_erd_ers(left_hand_epochs_p1_post_test, ref_left_post_test, fs, 13, 30)
beta_erd_right_post_test = calculate_erd_ers(right_hand_epochs_p1_post_test, ref_right_post_test, fs, 13, 30)

c:\Users\rodri\OneDrive\Documentos\g.tec hackaton\Hackaton_Rehab_Data_Analysis\processing.py:76: UserWarning: nperseg=512 is greater than signal length max(len(x), len(y)) = 256, using nperseg = 256
  f, Pxx = welch(epoch[:, ch], fs=fs, nperseg=fs*2)
